# Task 3: Exploratory Charts & Scope of Proof (What the Charts Do and Do Not Prove)
**Intern:** Avnika Gehlot | **Program:** SWYNEX Bioinformatics Internship | **Date:** November 23, 2026

## Objective
Visualize sequence quality, GC content distribution, and GC-vs-quality correlation across the `1,000 x 100 bp` FASTQ-style sequencing dataset analyzed in Task 2, and critically evaluate **what each chart proves** versus **what it does not prove**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# Reproduce the exact Task 2 FASTQ-style dataset (seed=42, 1,000 reads x 100 bp)
rng = np.random.default_rng(42)
n_reads, read_len = 1000, 100
bases = np.array(['A', 'T', 'G', 'C'])
read_gc_targets = np.clip(rng.normal(0.50, 0.08, size=n_reads), 0.25, 0.78)
positions = np.arange(1, read_len + 1)
mean_q_profile = 37.5 - 0.05 * positions - 0.00085 * (positions ** 2)

phred_matrix = np.zeros((n_reads, read_len), dtype=int)
gc_pct_list = []

for i in range(n_reads):
    gc_p = read_gc_targets[i]
    at_p = 1.0 - gc_p
    seq = rng.choice(bases, size=read_len, p=[at_p/2, at_p/2, gc_p/2, gc_p/2])
    gc_pct_list.append(((seq == 'G') | (seq == 'C')).mean() * 100.0)
    read_offset = rng.normal(0, 2.2)
    phred_matrix[i, :] = np.clip(np.round(rng.normal(mean_q_profile + read_offset, 2.5)), 2, 40).astype(int)

gc_pct_arr = np.array(gc_pct_list)
mean_read_q = phred_matrix.mean(axis=1)
r_val, p_val = stats.pearsonr(gc_pct_arr, mean_read_q)
print(f'Mean GC: {gc_pct_arr.mean():.2f}% | Q20: {(phred_matrix >= 20).mean()*100:.2f}% | Q30: {(phred_matrix >= 30).mean()*100:.2f}% | Pearson r: {r_val:.3f} (p={p_val:.3f})')

In [ ]:
# Chart 1: Per-Base Phred Quality Score Across Read Positions (1-100 bp)
q_mean = phred_matrix.mean(axis=0)
q_median = np.median(phred_matrix, axis=0)
q_25, q_75 = np.percentile(phred_matrix, [25, 75], axis=0)
q_10, q_90 = np.percentile(phred_matrix, [10, 90], axis=0)

fig, ax = plt.subplots(figsize=(10, 5))
ax.axhspan(28, 42, color='#d4edda', alpha=0.55, label='High Quality Zone (Q >= 28)')
ax.axhspan(20, 28, color='#fff3cd', alpha=0.55, label='Acceptable Quality Zone (Q20-Q28)')
ax.axhspan(0, 20, color='#f8d7da', alpha=0.55, label='Low Quality Zone (Q < 20)')
ax.fill_between(positions, q_10, q_90, color='#4a90d9', alpha=0.22, label='10th-90th Percentile')
ax.fill_between(positions, q_25, q_75, color='#1f4e78', alpha=0.40, label='IQR (25th-75th)')
ax.plot(positions, q_median, color='#0b2e59', linewidth=2.0, label='Median Phred Score')
ax.plot(positions, q_mean, color='#d9534f', linestyle='--', linewidth=1.6, label='Mean Phred Score')
ax.set_title('Chart 1: Per-Base Phred Quality Score Across Read Position (1-100 bp)', fontweight='bold')
ax.set_xlabel('Position in Read (bp)')
ax.set_ylabel('Phred Quality Score (Q)')
ax.set_xlim(1, 100)
ax.set_ylim(10, 42)
ax.legend(loc='lower left', fontsize=8.5, ncol=2)
plt.tight_layout()
plt.savefig('Chart1_Per_Base_Quality.png', dpi=300)
plt.show()

In [ ]:
# Chart 2: Per-Read GC Content (%) Distribution
fig, ax = plt.subplots(figsize=(9, 5))
counts, bins, _ = ax.hist(gc_pct_arr, bins=25, density=True, color='#2b6cb0', edgecolor='white', alpha=0.82, label='Observed Read GC%')
mu, sd = gc_pct_arr.mean(), gc_pct_arr.std()
x_vals = np.linspace(bins.min(), bins.max(), 200)
ax.plot(x_vals, stats.norm.pdf(x_vals, mu, sd), color='#c53030', linewidth=2.2, label=f'Normal Fit (Mean={mu:.1f}%, SD={sd:.1f}%)')
ax.axvline(mu, color='#1a202c', linestyle='--', linewidth=1.4, label=f'Sample Mean ({mu:.2f}%)')
ax.set_title('Chart 2: Per-Read GC Content (%) Distribution Across 1,000 Reads', fontweight='bold')
ax.set_xlabel('GC Content (%)')
ax.set_ylabel('Probability Density')
ax.legend(loc='upper right')
plt.tight_layout()
plt.savefig('Chart2_GC_Content_Distribution.png', dpi=300)
plt.show()

In [ ]:
# Chart 3: GC Content vs. Mean Phred Quality Correlation
fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(gc_pct_arr, mean_read_q, alpha=0.45, s=22, color='#2f855a', label='Individual Reads (n = 1,000)')
slope, intercept = np.polyfit(gc_pct_arr, mean_read_q, 1)
x_line = np.linspace(gc_pct_arr.min(), gc_pct_arr.max(), 100)
ax.plot(x_line, slope * x_line + intercept, color='#1a202c', linewidth=2.0, label=f'Linear Trend (r = {r_val:.3f}, p = {p_val:.3f})')
ax.axhline(30, color='#c53030', linestyle=':', linewidth=1.3, label='Q30 Threshold (99.9% Base Call Accuracy)')
ax.set_title('Chart 3: Per-Read GC Content vs. Mean Phred Quality Score', fontweight='bold')
ax.set_xlabel('GC Content (%)')
ax.set_ylabel('Mean Read Phred Score (Q)')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('Chart3_GC_vs_Quality_Correlation.png', dpi=300)
plt.show()